# **Scraping Planos Masterlink**

_Página do provedor: https://masterlinktelecom.net/_

In [1]:
# Bibliotecas
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
from datetime import date
import re

## **1. Extração**

In [2]:
# Obter lista de planos
texto_html = open("../../data/scraped/masterlink.html", encoding="utf-8")
soup = BeautifulSoup(texto_html, "html.parser")
conteiner_planos = soup.find("div", id="planosnet")
planos = conteiner_planos.find_all("div", class_="e-con-full")
len(planos)

7

In [3]:
# Extrair informações dos planos
def raspar_plano(plano):
    velocidade = plano.find_all("h2")[0].text
    pre_preco = plano.find_all("h2")
    preco = ".".join([i.text for i in pre_preco])
    pre_sva = plano.find_all("span", class_="elementor-icon-list-text")
    svas = ", ".join([i.text for i in pre_sva])
    return {"velocidade": velocidade,
            "preco": preco,
            "svas": svas}

dados = []

for i in planos:
    dados_plano = raspar_plano(i)
    dados.append(dados_plano)

df = pd.DataFrame(dados)
df = df[df["svas"] != ""]
df

,velocidade,preco,svas
0,500 MEGA,"500 MEGA.R$.109,.,90./mês","100% Fibra Óptica, WIFI 6.0 incluso , IPV6 inc..."
2,600 MEGA,"600 MEGA.R$.119.,90./mês","100% Fibra Óptica, WIFI 6.0 incluso, IPV6 inc..."
4,650 MEGA,"650 MEGA.+ 1 Ponto\n MESH 6.0.R$.139.,90./mês","100% Fibra Óptica, WIFI 6.0 incluso , IPV6 inc..."


## **2. Tratamento**

In [4]:
df_tratado = df.copy()

# Data e provedor
df_tratado["DATA"] = date.today().strftime("%Y-%m-%d")
df_tratado["ISP"] = "Masterlink"

# Velocidade
df_tratado["VELOCIDADE"] = df_tratado["velocidade"].str.extract(r"(\d+\.?\d*)")

# PRECO
df_tratado["PRECO"] = (
    df_tratado["preco"].str.split("$").str[1]
    .str.replace(r"\D", "", regex=True)
    .astype(float)
) / 100

# SVAs
df_tratado["SVAs"] = df_tratado["svas"]

#Seleção de colunas
df_tratado = df_tratado[["DATA", "ISP", "VELOCIDADE", "SVAs", "PRECO"]]

df_tratado

,DATA,ISP,VELOCIDADE,SVAs,PRECO
0,2026-05-12,Masterlink,500,"100% Fibra Óptica, WIFI 6.0 incluso , IPV6 inc...",109.9
2,2026-05-12,Masterlink,600,"100% Fibra Óptica, WIFI 6.0 incluso, IPV6 inc...",119.9
4,2026-05-12,Masterlink,650,"100% Fibra Óptica, WIFI 6.0 incluso , IPV6 inc...",139.9


## **3. Carga**

In [5]:
# Salvando
provedor = "MASTERLINK.csv"
nome_arq = df_tratado["DATA"].max() + "-" + provedor

# Na rede Valenet
pasta = r"\\fileserver\arquivos\Estrategia Comercial\2. POWER BI\ECM - Radar Mercado\ScrapingPlanos"
df_tratado.to_csv(fr"{pasta}\{nome_arq}", index=False)

# Neste Workspace
df_tratado.to_csv(f"../../data/flat/{provedor}", index=False)

print(f"🟢 {nome_arq} salvo!")

🟢 2026-05-12-MASTERLINK.csv salvo!
